# SQL Hesabatı

| Cədvəl | Sual |
|---|---|
| `dim_customers` | A1: Müştərilər kimlərdir? |
| `fact_orders` | A2: Çatdırılma necə performans göstərir?<br>A3: Müştərilər necə ödəniş edir? |
| `fact_order_items` | A4: Gəlir və sifarişlər zamanla necə dəyişir? |
| `mart_customer_rfm` | A5: Ən yaxşı və ən zəif müştərilər kimlərdir? (RFM) |
| `dim_geography` | B1: Müştərilər və satıcılar coğrafi olaraq harada yerləşir? |
| `dim_products` | B2: Hansı məhsullar və kateqoriyalar mövcuddur? |
| `dim_sellers` | B3: Satıcılar kimlərdir və haradadır? |
| `dim_date` | B4: Hər günün təqvim, bayram, məzənnə və hava konteksti nədir? |
| `fact_reviews` | B5: Müştərilər sifarişləri necə qiymətləndirir? |
| `mart_seller_performance` | B6: Ən yaxşı və ən zəif satıcılar kimlərdir? |
| `mart_daily_sales_weather` | B7: Hava São Paulo-da gündəlik satışlara necə təsir edir? |

## A0. Setup

In [4]:
# Importing libraries
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

# db.py lives in notebooks/, two levels above this notebook's folder
# (adjust .parent chain if this notebook is placed in a different folder)
NOTEBOOKS_DIR = Path.cwd().parent.parent
sys.path.append(str(NOTEBOOKS_DIR))

from db import get_engine
engine = get_engine()


In [5]:
def q(sql):
    """Run a SELECT and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def run(sql):
    """Run CREATE/DROP/INSERT and commit."""
    with engine.begin() as conn:
        conn.execute(text(sql))

---

## A1. gold.dim_customers
- **Explore:** customers vs unique people, repeat-buyer share, customers per state.
- **Base:** `silver.customers`; **Joins:** none.
- **Columns:** `customer_id`, `customer_unique_id`, `customer_zip_code_prefix`, `customer_city`, `customer_state`.
- **Note:** a real person is `customer_unique_id` (one person can have several `customer_id` values). Use `DISTINCTCOUNT(customer_unique_id)` in Power BI for customer counts.

In [8]:
# Rebuild gold.dim_customers (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_customers"))
    conn.execute(text("""
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix,
            customer_city,
            customer_state
        INTO gold.dim_customers
        FROM silver.customers
    """))

---

## A2. gold.fact_orders
- **Explore:** share of delivered orders, delivery days, late share by state.
**`gold.fact_orders` spec**
- **Columns:** `order_id`, `customer_id`, `order_status`, `order_date` (DATE), `approved_date`, `delivered_carrier_date`, `delivered_customer_date`, `estimated_delivery_date`, `delivery_days`, `delay_days`, `is_late` (BIT), `total_payment`, `payment_installments_max`, `main_payment_type`.
- **Logic:** `delivery_days` = delivered date minus purchase date; `delay_days` = delivered date minus estimated date; `is_late` = `delay_days > 0`. Delivery columns stay `NULL` for undelivered orders (`is_late` is NULL, not 0).

In [15]:
# Step 1: one row per order with total payment and max installments
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_pay"))
    conn.execute(text("""
        SELECT order_id,
               SUM(payment_value)        AS total_payment,
               MAX(payment_installments) AS payment_installments_max
        INTO gold._tmp_pay
        FROM silver.order_payments
        GROUP BY order_id
    """))

q("SELECT COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders FROM gold._tmp_pay")

,rows_,orders
0,99440,99440


In [16]:
# Step 2: main payment type = type with the highest value in the order
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_main_type"))
    conn.execute(text("""
        WITH t AS (
            SELECT order_id, payment_type,
                   ROW_NUMBER() OVER (
                       PARTITION BY order_id
                       ORDER BY SUM(payment_value) DESC, payment_type
                   ) AS rn
            FROM silver.order_payments
            GROUP BY order_id, payment_type
        )
        SELECT order_id, payment_type AS main_payment_type
        INTO gold._tmp_main_type
        FROM t
        WHERE rn = 1
    """))

q("SELECT COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders FROM gold._tmp_main_type")

,rows_,orders
0,99440,99440


In [17]:
# Step 3: orders with delivery_days and delay_days (NULL if not delivered)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_base"))
    conn.execute(text("""
        SELECT order_id,
               customer_id,
               order_status,
               CAST(order_purchase_timestamp AS DATE) AS order_date,
               order_approved_at                      AS approved_date,
               order_delivered_carrier_date           AS delivered_carrier_date,
               order_delivered_customer_date          AS delivered_customer_date,
               order_estimated_delivery_date          AS estimated_delivery_date,
               CASE WHEN order_delivered_customer_date IS NOT NULL
                    THEN DATEDIFF(day, CAST(order_purchase_timestamp AS DATE),
                                       CAST(order_delivered_customer_date AS DATE)) END AS delivery_days,
               CASE WHEN order_delivered_customer_date IS NOT NULL
                         AND order_estimated_delivery_date IS NOT NULL
                    THEN DATEDIFF(day, CAST(order_estimated_delivery_date AS DATE),
                                       CAST(order_delivered_customer_date AS DATE)) END AS delay_days
        INTO gold._tmp_base
        FROM silver.orders
    """))

q("SELECT COUNT(*) AS rows_ FROM gold._tmp_base")

,rows_
0,99441


In [18]:
# Step 4: final table = base + payment total + main payment type
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_orders"))
    conn.execute(text("""
        SELECT b.order_id,
               b.customer_id,
               b.order_status,
               b.order_date,
               b.approved_date,
               b.delivered_carrier_date,
               b.delivered_customer_date,
               b.estimated_delivery_date,
               b.delivery_days,
               b.delay_days,
               -- NULL (not 0) when delay_days is unknown
               CAST(CASE WHEN b.delay_days IS NULL THEN NULL
                         WHEN b.delay_days > 0 THEN 1
                         ELSE 0 END AS BIT)  AS is_late,
               p.total_payment,
               p.payment_installments_max,
               m.main_payment_type
        INTO gold.fact_orders
        FROM gold._tmp_base b
        LEFT JOIN gold._tmp_pay p       ON p.order_id = b.order_id
        LEFT JOIN gold._tmp_main_type m ON m.order_id = b.order_id
    """))

q("""
SELECT (SELECT COUNT(*) FROM silver.orders)    AS silver_rows,
       (SELECT COUNT(*) FROM gold.fact_orders) AS gold_rows
""")

,silver_rows,gold_rows
0,99441,99441


In [19]:
# Step 5: drop the staging tables
with engine.begin() as conn:
    for t in ["_tmp_pay", "_tmp_main_type", "_tmp_base"]:
        conn.execute(text(f"DROP TABLE IF EXISTS gold.{t}"))

---

## A4. gold.fact_order_items
- **Explore:** monthly revenue and orders on silver, revenue by order status, orphan check.
- **Joins:** `LEFT JOIN silver.orders` on `order_id` (one row per order, so no duplication) for `customer_id`, `order_date`, `order_status`.
- **Columns:** `order_id`, `order_item_id`, `product_id`, `seller_id`, `customer_id`, `order_date`, `order_status`, `price`, `freight_value`, `item_total` (price + freight).


In [27]:
# Build gold.fact_order_items (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_order_items"))
    conn.execute(text("""
        SELECT i.order_id,
               i.order_item_id,
               i.product_id,
               i.seller_id,
               o.customer_id,
               CAST(o.order_purchase_timestamp AS DATE) AS order_date,
               o.order_status,
               i.price,
               i.freight_value,
               i.price + i.freight_value                AS item_total
        INTO gold.fact_order_items
        FROM silver.order_items i
        LEFT JOIN silver.orders o ON o.order_id = i.order_id
    """))

---

## A5. gold.mart_customer_rfm (RFM)
- **Joins:** `gold.fact_orders` + `gold.dim_customers` (to get `customer_unique_id`) + `gold.fact_order_items` aggregated to one row per `order_id` first.
- **Columns:** `customer_unique_id`, `recency_days`, `frequency`, `monetary`, `r_score` (1-5), `f_score` (1-3), `m_score` (1-5), `segment`.
- **Logic:** reference date = last delivered order + 1 day. R and M use `NTILE(5)`. F: 1 order = 1, 2 orders = 2, 3+ = 3.
- **Segments:** Champions (f >= 2 and r >= 4), Loyal (f >= 2), New high value (r >= 4 and m >= 4), New (r >= 4), Lost (r = 1), Needs attention (rest).

In [ ]:
# Build gold.mart_customer_rfm in one query (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_customer_rfm"))
    conn.execute(text("""
        WITH items AS (
            -- Aggregate items to one row per order BEFORE joining
            SELECT order_id, SUM(price) AS order_value
            FROM gold.fact_order_items
            GROUP BY order_id
        ),
        ord AS (
            SELECT c.customer_unique_id, f.order_id, f.order_date,
                   COALESCE(i.order_value, 0) AS order_value
            FROM gold.fact_orders f
            JOIN gold.dim_customers c ON c.customer_id = f.customer_id
            LEFT JOIN items i         ON i.order_id = f.order_id
            WHERE f.order_status = 'delivered'
        ),
        base AS (
            -- Reference date = day after the last order in the data
            SELECT customer_unique_id,
                   DATEDIFF(day, MAX(order_date), (SELECT DATEADD(day, 1, MAX(order_date)) FROM ord)) AS recency_days,
                   COUNT(DISTINCT order_id) AS frequency,
                   SUM(order_value)         AS monetary
            FROM ord
            GROUP BY customer_unique_id
        ),
        s AS (
            SELECT *,
                   NTILE(5) OVER (ORDER BY recency_days DESC, customer_unique_id) AS r_score,
                   CASE WHEN frequency >= 3 THEN 3 WHEN frequency = 2 THEN 2 ELSE 1 END AS f_score,
                   NTILE(5) OVER (ORDER BY monetary ASC, customer_unique_id)      AS m_score
            FROM base
        )
        SELECT customer_unique_id, recency_days, frequency, monetary,
               r_score, f_score, m_score,
               CASE WHEN f_score >= 2 AND r_score >= 4 THEN 'Champions'
                    WHEN f_score >= 2                  THEN 'Loyal'
                    WHEN r_score >= 4 AND m_score >= 4 THEN 'New high value'
                    WHEN r_score >= 4                  THEN 'New'
                    WHEN r_score = 1                   THEN 'Lost'
                    ELSE 'Needs attention' END         AS segment
        INTO gold.mart_customer_rfm
        FROM s
    """))

## 1. `gold.dim_geography`
#Unikal poct prefiksi üçün bir sətrdə digər sütunları qruplaşdırırıq təkrarlanan məlumatları ümümləşdiririk

In [ ]:

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_geography"))

    # Hər poçt prefiksi üçün bir sətir yaradırıq
    conn.execute(text("""
        SELECT
            zip_code_prefix,       -- Poçt prefiksi
            AVG(lat) AS lat,        -- Orta enlik koordinatı
            AVG(lng) AS lng,        -- Orta uzunluq koordinatı
            MAX(city) AS city,      -- Şəhər adı
            MAX(state) AS state     -- Ştat kodu
        INTO gold.dim_geography    -- Yeni Gold cədvəli yaradırıq
        FROM silver.geolocation   
        GROUP BY zip_code_prefix  
    """))
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT zip_code_prefix) AS unique_prefixes
    FROM gold.dim_geography
""", engine)

print(df.to_string(index=False))

## 2. `gold.dim_products`
Burada məhsul məlumatları ilə tərcümə olunmuş kateqoriya adlarını birləşdirib bir cədvələ yığırıq

In [ ]:

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_products"))

    conn.execute(text("""
        SELECT
            p.product_id,
            p.category_name,
            COALESCE(t.category_name_english, p.category_name, 'unknown') AS category_name_english,
            p.name_length,
            p.description_length,
            p.photos_qty,
            p.weight_g,
            p.length_cm,
            p.height_cm,
            p.width_cm
        INTO gold.dim_products
        FROM silver.products p
        LEFT JOIN silver.category_translation t
            ON p.category_name = t.category_name
    """))


## 3. `gold.dim_sellers`
# Satıcı məlumatlarını Gold cədvəlinə köçürürük

In [ ]:

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_sellers"))

    conn.execute(text("""
        SELECT
            seller_id,
            zip_code_prefix,
            city,
            state
        INTO gold.dim_sellers
        FROM silver.sellers
    """))
# Satıcı sayını və təkrarlanan ID-ləri yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT seller_id) AS unique_sellers
    FROM gold.dim_sellers
""", engine)

df

In [ ]:
## 4. `gold.dim_date`
# Tarixlər cədvəlini yaradırıq
with engine.begin() as conn:  # Verilənlər bazasına qoşuluruq və əməliyyatları idarə edirik
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_date"))  # Əvvəlki cədvəl varsa, silirik

    conn.execute(text("""
        WITH dates AS (  -- Tarixləri ardıcıl yaratmaq üçün müvəqqəti sorğu hazırlayırıq
            SELECT CAST('2016-01-01' AS DATE) AS [date]  -- Başlanğıc tarixini 1 yanvar 2016 təyin edirik

            UNION ALL  -- Yeni tarixləri əvvəlki nəticəyə əlavə edirik

            SELECT DATEADD(DAY, 1, [date])  -- Əvvəlki tarixə 1 gün əlavə edirik
            FROM dates  -- Əvvəl yaratdığımız tarixlərdən istifadə edirik
            WHERE [date] < '2018-12-31'  -- Son tarixə çatana qədər davam edirik
        )
        SELECT
            [date],  -- Tarixi göstəririk
            YEAR([date]) AS [year],  -- Tarixin ilini çıxarırıq
            DATEPART(QUARTER, [date]) AS [quarter],  -- İlin hansı rübü olduğunu tapırıq
            MONTH([date]) AS [month],  -- Ayın nömrəsini çıxarırıq
            DATENAME(MONTH, [date]) AS month_name,  -- Ayın adını çıxarırıq
            DATEPART(ISO_WEEK, [date]) AS [week],  -- ISO standartına görə həftənin nömrəsini tapırıq
            ((DATEPART(WEEKDAY, [date]) + @@DATEFIRST - 2) % 7) + 1 AS day_of_week,  -- Həftənin gününü bazar ertəsi=1, bazar=7 formasına gətiririk
            CASE
                WHEN ((DATEPART(WEEKDAY, [date]) + @@DATEFIRST - 2) % 7) + 1 IN (6, 7)  -- Gün şənbə və ya bazardırsa
                THEN 1 ELSE 0  -- Həftəsonudursa 1, deyilsə 0 yazırıq
            END AS is_weekend  -- Bu sütunun adını is_weekend qoyuruq
        INTO gold.dim_date  -- Nəticəni gold.dim_date adlı yeni cədvələ yazırıq
        FROM dates  -- Yaratdığımız tarix siyahısından məlumat götürürük
        OPTION (MAXRECURSION 0)  -- Təkrarlanan sorğu üçün maksimum addım məhdudiyyətini götürürük
    """))







In [ ]:
# Bayram günlərini göstərən sütunu cədvələ əlavə edirik
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD is_holiday BIT NULL
    """))


In [ ]:
# Milli bayramları tarix cədvəli ilə uyğunlaşdırırıq
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET is_holiday = CASE
            WHEN h.holiday_date IS NOT NULL THEN 1
            ELSE 0
        END
        FROM gold.dim_date d
        LEFT JOIN silver.holidays h
            ON d.[date] = h.holiday_date
            AND h.is_national = 1
    """))

In [ ]:
#Dollar məzənnəsini saxlamaq üçün sütun əlavə edirik
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD usd_per_brl DECIMAL(10,5) NULL
    """))

In [ ]:
#Hər tarix üçün həmin günə və ya əvvəlki ən yaxın məzənnəni götürürük
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET usd_per_brl = (
            SELECT TOP 1 e.rate_usd
            FROM silver.exchange_rates e
            WHERE e.rate_date <= d.[date]
            ORDER BY e.rate_date DESC
        )
        FROM gold.dim_date d
    """))

In [ ]:
#Hava məlumatlarını saxlamaq üçün 3 sütun əlavə edirik
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD sp_temp_max FLOAT NULL,
            sp_temp_min FLOAT NULL,
            sp_precipitation FLOAT NULL
    """))

In [ ]:

#Hava məlumatlarını tarixlərə uyğunlaşdırıb Gold cədvəlinə yazırıq
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET
            sp_temp_max = w.temp_max,
            sp_temp_min = w.temp_min,
            sp_precipitation = w.precipitation
        FROM gold.dim_date d
        LEFT JOIN silver.weather w
            ON d.[date] = w.weather_date
    """))

## 5. `gold.fact_reviews`

In [ ]:
# Rəy məlumatlarını Gold fakt cədvəlinə köçürürük
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_reviews"))

    conn.execute(text("""
        SELECT
            r.review_id,
            r.order_id,
            r.review_score,
            r.comment_title,
            r.comment_message,
            r.creation_date,
            r.answer_timestamp
        INTO gold.fact_reviews
        FROM silver.reviews r
        INNER JOIN silver.orders o
            ON r.order_id = o.order_id
    """))
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT CONCAT(review_id, order_id)) AS unique_combinations
    FROM gold.fact_reviews
""", engine)

display(df)


## 6. `gold.mart_seller_performance`


In [ ]:
# Satıcı performansı mart cədvəlini yaradırıq
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_seller_performance"))

    conn.execute(text("""
        SELECT
            s.seller_id,
            s.zip_code_prefix,
            s.city,
            s.state,
            COALESCE(x.total_orders, 0) AS total_orders,
            COALESCE(x.items_sold, 0) AS items_sold,
            COALESCE(x.total_sales, 0) AS total_sales,
            COALESCE(x.total_freight, 0) AS total_freight
        INTO gold.mart_seller_performance
        FROM silver.sellers s
        LEFT JOIN (
            SELECT
                oi.seller_id,
                COUNT(DISTINCT oi.order_id) AS total_orders,
                COUNT(*) AS items_sold,
                SUM(oi.price) AS total_sales,
                SUM(oi.freight_value) AS total_freight
            FROM silver.order_items oi
            INNER JOIN silver.orders o
                ON oi.order_id = o.order_id
            WHERE o.order_status = 'delivered'
            GROUP BY oi.seller_id
        ) x
            ON s.seller_id = x.seller_id
    """))

## 7. `gold.mart_daily_sales_weather`


In [ ]:

# Gündəlik satış və hava məlumatlarını Gold mart cədvəlinə yazırıq
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_daily_sales_weather"))

    conn.execute(text("""
        SELECT
            d.[date] AS sales_date,
            d.[year],
            d.[month],
            d.is_weekend,
            d.is_holiday,
            d.usd_per_brl,
            d.sp_temp_max,
            d.sp_temp_min,
            d.sp_precipitation,
            COALESCE(s.total_orders, 0) AS total_orders,
            COALESCE(s.items_sold, 0) AS items_sold,
            COALESCE(s.total_sales, 0) AS total_sales,
            COALESCE(s.total_freight, 0) AS total_freight
        INTO gold.mart_daily_sales_weather
        FROM gold.dim_date d
        LEFT JOIN (
            SELECT
                CAST(o.order_purchase_timestamp AS DATE) AS sales_date,
                COUNT(DISTINCT o.order_id) AS total_orders,
                COUNT(*) AS items_sold,
                SUM(oi.price) AS total_sales,
                SUM(oi.freight_value) AS total_freight
            FROM silver.orders o
            INNER JOIN silver.order_items oi
                ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered'
            GROUP BY CAST(o.order_purchase_timestamp AS DATE)
        ) s
            ON d.[date] = s.sales_date
    """))
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM gold.mart_daily_sales_weather
    ORDER BY sales_date
""", engine)

df